# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_03 — Transaction Costs, Slippage, and Latency

### Research question

How do transaction costs, bid/ask spread, slippage, market impact, latency, participation constraints, and fill uncertainty change the true performance and capacity of a trading strategy?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
```

The vectorized engine handled simple cost accounting. The event-driven engine handled event order and fills. This notebook isolates the **execution cost model** itself.

It covers:

1. cost taxonomy;
2. half-spread cost;
3. commissions and fees;
4. fixed slippage;
5. volatility-scaled slippage;
6. participation-rate constraints;
7. linear impact model;
8. square-root impact model;
9. implementation shortfall;
10. arrival price versus fill price;
11. latency-induced price drift;
12. alpha decay under latency;
13. bid/ask bounce;
14. partial fills and opportunity cost;
15. capacity curves;
16. cost sensitivity grids;
17. turnover-aware net returns;
18. before-cost versus after-cost strategy performance;
19. TCA reporting;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> Backtests do not fail only because the alpha is fake. They also fail because the alpha is too small, too slow, too expensive, or too capacity-constrained to trade.

## 1. Cost taxonomy

A realistic trading cost model separates:

### Explicit costs

- commissions;
- exchange fees;
- clearing fees;
- taxes.

### Implicit costs

- bid/ask spread;
- slippage;
- market impact;
- delay cost;
- opportunity cost from unfilled quantity;
- adverse selection.

A simple decomposition:

$$
\begin{aligned}
TotalCost &= SpreadCost \\
&\quad + Commission \\
&\quad + Impact \\
&\quad + DelayCost \\
&\quad + OpportunityCost
\end{aligned}
$$

This notebook implements each component in a simplified but auditable way.

## 2. Implementation shortfall

Implementation shortfall compares execution to the arrival decision price.

For a buy order:

$$
IS = \frac{P_{fill}-P_{arrival}}{P_{arrival}}
$$

For a sell order:

$$
IS = \frac{P_{arrival}-P_{fill}}{P_{arrival}}
$$

A signed version using trade direction $q$ is:

$$
IS = \operatorname{sign}(q) \frac{P_{fill}-P_{arrival}}{P_{arrival}}
$$

where positive means cost.

Implementation shortfall is the core language of transaction-cost analysis.

## 3. Market impact models

### Linear impact

$$
impact = \eta \cdot participation
$$

Simple but often too aggressive at high participation.

### Square-root impact

$$
impact = Y\sigma \sqrt{\frac{|Q|}{ADV}}
$$

where:

- $Y$ is an impact coefficient;
- $\sigma$ is volatility;
- $Q$ is trade notional;
- $ADV$ is average daily volume.

The square-root model is widely used as an empirical approximation for large orders.

## 4. Latency and alpha decay

If a signal decays with half-life $h$, then after delay $\Delta t$:

$$
signal_{\Delta t} = signal_0 e^{-\ln(2)\Delta t/h}
$$

Execution delay reduces expected alpha.

A strategy can look profitable with immediate fills and fail once realistic latency is introduced.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class CostLatencyConfig:
    n_days: int = 600
    minutes_per_day: int = 78
    seed: int = 42
    annualisation: int = 252
    initial_capital: float = 5_000_000.0
    rebalance_days: int = 5
    target_gross: float = 1.20
    max_abs_weight: float = 0.25
    commission_bps: float = 0.35
    exchange_fee_bps: float = 0.05
    fixed_slippage_bps: float = 1.0
    volatility_slippage_multiplier: float = 0.08
    linear_impact_coefficient_bps: float = 18.0
    sqrt_impact_coefficient: float = 0.65
    max_participation: float = 0.10
    signal_half_life_minutes: float = 90.0
    latency_grid_minutes: tuple = (0, 1, 2, 5, 10, 20, 40, 80)
    participation_grid: tuple = (0.01, 0.025, 0.05, 0.10, 0.20, 0.35)
    cvar_alpha: float = 0.95

config = CostLatencyConfig()
config

## 6. Simulate daily and intraday market data

We simulate:

- daily close-to-close returns;
- intraday mid prices;
- bid/ask spreads;
- intraday volume;
- average daily volume;
- realised volatility;
- stress regimes.

The assets have different liquidity profiles so cost estimates differ materially.

In [ ]:
def simulate_cost_research_data(config: CostLatencyConfig):
    rng = np.random.default_rng(config.seed)
    assets = ["US_EQ", "EU_EQ", "EM_EQ", "US_BOND", "GOLD", "OIL", "COPPER", "BTC_PROXY"]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "BTC_PROXY": "crypto",
    }

    base_spread_bps = pd.Series({
        "US_EQ": 1.0,
        "EU_EQ": 1.5,
        "EM_EQ": 4.5,
        "US_BOND": 0.7,
        "GOLD": 2.0,
        "OIL": 3.2,
        "COPPER": 3.8,
        "BTC_PROXY": 14.0,
    })

    adv_usd = pd.Series({
        "US_EQ": 8e9,
        "EU_EQ": 4e9,
        "EM_EQ": 1.5e9,
        "US_BOND": 10e9,
        "GOLD": 3e9,
        "OIL": 2e9,
        "COPPER": 1e9,
        "BTC_PROXY": 0.8e9,
    })

    ann_vol = pd.Series({
        "US_EQ": 0.16,
        "EU_EQ": 0.18,
        "EM_EQ": 0.24,
        "US_BOND": 0.06,
        "GOLD": 0.18,
        "OIL": 0.32,
        "COPPER": 0.28,
        "BTC_PROXY": 0.70,
    })

    ann_mu = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.06,
        "EM_EQ": 0.08,
        "US_BOND": 0.025,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "BTC_PROXY": 0.120,
    })

    dates = pd.bdate_range("2020-01-01", periods=config.n_days)

    factor_corr = np.array([
        [1.00, -0.20, 0.25, 0.35],
        [-0.20, 1.00, 0.10, -0.20],
        [0.25, 0.10, 1.00, 0.20],
        [0.35, -0.20, 0.20, 1.00],
    ])
    factor_vol_daily = np.array([0.010, 0.004, 0.012, 0.030])
    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=["market", "rates", "commodity", "crypto"])
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ"], "market"] = [1.00, 0.95, 1.20]
    loadings.loc["US_BOND", "rates"] = 1.00
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.35
    loadings.loc["GOLD", "rates"] = 0.25

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    daily_returns = pd.DataFrame(index=dates, columns=assets, dtype=float)
    daily_spread_bps = pd.DataFrame(index=dates, columns=assets, dtype=float)
    daily_volume_usd = pd.DataFrame(index=dates, columns=assets, dtype=float)

    transition = np.array([[0.985, 0.015], [0.060, 0.940]])

    for t, date in enumerate(dates):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_mult = 1.0 if regime[t] == 0 else 2.6
        f = rng.multivariate_normal(np.zeros(4), factor_cov * vol_mult**2)

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.030, 0.090)
            f[1] += rng.uniform(0.003, 0.018)
            f[3] -= rng.uniform(0.060, 0.200)
        elif u < 0.014:
            stress_type[t] = "commodity_shock"
            f[2] += rng.choice([-1, 1]) * rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)
        elif u < 0.018:
            stress_type[t] = "liquidity_gap"

        eps = rng.standard_normal(len(assets)) * ann_vol.values / np.sqrt(config.annualisation) * 0.35 * vol_mult
        daily_returns.iloc[t] = ann_mu.values / config.annualisation + loadings.to_numpy() @ f + eps

        spread_noise = rng.lognormal(0.0, 0.25, len(assets))
        liquidity_multiplier = 1.0 + 1.0 * regime[t] + (2.0 if stress_type[t] == "liquidity_gap" else 0.0)
        daily_spread_bps.iloc[t] = base_spread_bps.values * spread_noise * liquidity_multiplier

        volume_noise = rng.lognormal(0.0, 0.45, len(assets))
        volume_multiplier = 1.0 - 0.20 * regime[t] - (0.35 if stress_type[t] == "liquidity_gap" else 0.0)
        daily_volume_usd.iloc[t] = adv_usd.values * volume_noise * max(volume_multiplier, 0.25)

    prices = 100.0 * np.exp(np.log1p(daily_returns.astype(float)).cumsum())

    # Intraday data from daily returns, mainly for latency experiments.
    intraday_index = []
    intraday_mid = []
    intraday_spread = []
    intraday_volume = []

    for date in dates:
        start = pd.Timestamp(date.date()) + pd.Timedelta(hours=9, minutes=30)
        daily_ret = daily_returns.loc[date].astype(float)
        close_to_close_path = np.linspace(0, 1, config.minutes_per_day)

        for m in range(config.minutes_per_day):
            ts = start + pd.Timedelta(minutes=5 * m)
            intraday_index.append(ts)

            noise = rng.standard_normal(len(assets)) * ann_vol.values / np.sqrt(config.annualisation * config.minutes_per_day) * 0.45
            mid_values = prices.loc[date].values * np.exp(close_to_close_path[m] * daily_ret.values + noise)
            intraday_mid.append(mid_values)

            intraday_spread.append(daily_spread_bps.loc[date].values * rng.lognormal(0, 0.10, len(assets)))
            intraday_volume.append(daily_volume_usd.loc[date].values / config.minutes_per_day * rng.lognormal(0, 0.45, len(assets)))

    intraday_mid = pd.DataFrame(intraday_mid, index=pd.DatetimeIndex(intraday_index, name="timestamp"), columns=assets)
    intraday_spread = pd.DataFrame(intraday_spread, index=intraday_mid.index, columns=assets)
    intraday_volume = pd.DataFrame(intraday_volume, index=intraday_mid.index, columns=assets)

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "base_spread_bps": [base_spread_bps[a] for a in assets],
        "adv_usd": [adv_usd[a] for a in assets],
        "ann_vol_assumption": [ann_vol[a] for a in assets],
        "ann_mu_assumption": [ann_mu[a] for a in assets],
    })

    regime_info = pd.DataFrame({
        "regime": regime,
        "stress_type": stress_type,
    }, index=dates)

    return daily_returns.astype(float), prices, daily_spread_bps.astype(float), daily_volume_usd.astype(float), intraday_mid, intraday_spread.astype(float), intraday_volume.astype(float), metadata, regime_info

daily_returns, prices, daily_spread_bps, daily_volume_usd, intraday_mid, intraday_spread, intraday_volume, metadata, regime_info = simulate_cost_research_data(config)
assets = metadata["asset"].tolist()

daily_returns.head(), metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in assets:
    plt.plot(prices.index, prices[asset], label=asset)
plt.title("Synthetic Daily Prices")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(regime_info.index, regime_info["regime"], drawstyle="steps-post")
plt.title("Liquidity / Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 7. Research signal and target trades

We generate a cross-sectional signal and convert it into target weights.

The purpose is not alpha quality. The purpose is to create realistic turnover that the cost model can evaluate.

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sigma = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sigma, axis=0).fillna(0.0)

def make_signal(prices, daily_returns):
    momentum_63 = prices.pct_change(63)
    reversal_10 = -prices.pct_change(10)
    vol_21 = daily_returns.rolling(21).std() * np.sqrt(config.annualisation)

    signal = (
        0.55 * cross_sectional_zscore(momentum_63)
        + 0.25 * cross_sectional_zscore(reversal_10)
        + 0.20 * cross_sectional_zscore(-vol_21)
    )

    return signal.clip(-3, 3).fillna(0.0)

def signal_to_weights(signal, returns, gross_target, max_abs_weight):
    vol = returns.rolling(63).std().shift(1)
    vol = vol.fillna(returns.expanding().std().shift(1)).fillna(returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(gross_target).fillna(0.0)
    weights = weights.clip(-max_abs_weight, max_abs_weight)

    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(gross_target).fillna(0.0)
    return weights

def apply_rebalance(weights, step):
    scheduled = weights.copy() * np.nan
    scheduled.iloc[::step] = weights.iloc[::step]
    return scheduled.ffill().fillna(0.0)

signal = make_signal(prices, daily_returns)
target_weights = apply_rebalance(
    signal_to_weights(signal, daily_returns, config.target_gross, config.max_abs_weight),
    config.rebalance_days
)

held_weights = target_weights.shift(1).fillna(0.0)
delta_weights = held_weights.diff().fillna(held_weights)
trade_notional = delta_weights.abs() * config.initial_capital

target_weights.tail()

In [ ]:
plt.figure(figsize=(12, 5))
for asset in ["US_EQ", "US_BOND", "GOLD", "OIL", "BTC_PROXY"]:
    plt.plot(target_weights.index, target_weights[asset], label=asset)
plt.title("Target Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend()
plt.show()

## 8. Cost model functions

We implement separate components:

1. half-spread;
2. commission and fees;
3. fixed slippage;
4. volatility-scaled slippage;
5. linear impact;
6. square-root impact;
7. delay cost;
8. opportunity cost.

Keeping components separate makes TCA explainable.

In [ ]:
def half_spread_cost_bps(spread_bps):
    return spread_bps / 2.0

def explicit_fee_cost_bps(config):
    return config.commission_bps + config.exchange_fee_bps

def fixed_slippage_cost_bps(config):
    return config.fixed_slippage_bps

def volatility_scaled_slippage_bps(realised_vol_ann, config):
    daily_vol_bps = realised_vol_ann / np.sqrt(config.annualisation) * 10000.0
    return config.volatility_slippage_multiplier * daily_vol_bps

def participation_rate(trade_notional, daily_volume_usd):
    return trade_notional / daily_volume_usd.replace(0.0, np.nan)

def linear_impact_bps(participation, config):
    return config.linear_impact_coefficient_bps * participation

def square_root_impact_bps(trade_notional, daily_volume_usd, realised_vol_ann, config):
    part = participation_rate(trade_notional, daily_volume_usd).clip(lower=0.0)
    daily_vol_bps = realised_vol_ann / np.sqrt(config.annualisation) * 10000.0
    return config.sqrt_impact_coefficient * daily_vol_bps * np.sqrt(part)

def alpha_decay_multiplier(latency_minutes, half_life_minutes):
    return np.exp(-np.log(2.0) * latency_minutes / half_life_minutes)

## 9. Build daily realised volatility and participation matrices

In [ ]:
realised_vol_ann = daily_returns.rolling(21).std().shift(1) * np.sqrt(config.annualisation)
realised_vol_ann = realised_vol_ann.fillna(daily_returns.expanding().std().shift(1) * np.sqrt(config.annualisation))
realised_vol_ann = realised_vol_ann.fillna(daily_returns.std() * np.sqrt(config.annualisation))

participation = participation_rate(trade_notional, daily_volume_usd).fillna(0.0)

participation_summary = pd.DataFrame({
    "asset": assets,
    "mean_participation": participation.mean().values,
    "p95_participation": participation.quantile(0.95).values,
    "max_participation": participation.max().values,
}).merge(metadata[["asset", "asset_class", "adv_usd"]], on="asset", how="left")

participation_summary.sort_values("p95_participation", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = participation_summary.sort_values("p95_participation")
plt.barh(plot_df["asset"], plot_df["p95_participation"])
plt.axvline(config.max_participation, linestyle="--", label="max participation guideline")
plt.title("95th Percentile Participation by Asset")
plt.xlabel("Participation")
plt.ylabel("Asset")
plt.legend()
plt.show()

## 10. Estimate cost matrices

All costs are estimated in basis points per traded notional.

Total cost per traded notional:

$$
\begin{aligned}
c_{i,t}^{bps} &= spread/2 \\
&\quad + fees \\
&\quad + slippage \\
&\quad + impact
\end{aligned}
$$

In [ ]:
spread_component = half_spread_cost_bps(daily_spread_bps)
fee_component = pd.DataFrame(explicit_fee_cost_bps(config), index=daily_returns.index, columns=assets)
fixed_slippage_component = pd.DataFrame(fixed_slippage_cost_bps(config), index=daily_returns.index, columns=assets)
vol_slippage_component = volatility_scaled_slippage_bps(realised_vol_ann, config)
linear_impact_component = linear_impact_bps(participation, config)
sqrt_impact_component = square_root_impact_bps(trade_notional, daily_volume_usd, realised_vol_ann, config)

total_cost_linear_bps = (
    spread_component
    + fee_component
    + fixed_slippage_component
    + vol_slippage_component
    + linear_impact_component
)

total_cost_sqrt_bps = (
    spread_component
    + fee_component
    + fixed_slippage_component
    + vol_slippage_component
    + sqrt_impact_component
)

cost_component_summary = pd.DataFrame({
    "component": [
        "half_spread",
        "fees",
        "fixed_slippage",
        "vol_scaled_slippage",
        "linear_impact",
        "sqrt_impact",
        "total_linear",
        "total_sqrt",
    ],
    "mean_bps": [
        spread_component.stack().mean(),
        fee_component.stack().mean(),
        fixed_slippage_component.stack().mean(),
        vol_slippage_component.stack().mean(),
        linear_impact_component.stack().mean(),
        sqrt_impact_component.stack().mean(),
        total_cost_linear_bps.stack().mean(),
        total_cost_sqrt_bps.stack().mean(),
    ],
    "p95_bps": [
        spread_component.stack().quantile(0.95),
        fee_component.stack().quantile(0.95),
        fixed_slippage_component.stack().quantile(0.95),
        vol_slippage_component.stack().quantile(0.95),
        linear_impact_component.stack().quantile(0.95),
        sqrt_impact_component.stack().quantile(0.95),
        total_cost_linear_bps.stack().quantile(0.95),
        total_cost_sqrt_bps.stack().quantile(0.95),
    ],
})

cost_component_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(cost_component_summary["component"], cost_component_summary["mean_bps"])
plt.title("Mean Cost Component bps per Traded Notional")
plt.xlabel("bps")
plt.ylabel("Component")
plt.show()

## 11. Net return under different cost assumptions

We compare:

1. no cost;
2. half-spread + fees;
3. linear impact model;
4. square-root impact model.

Cost in return units:

$$
cost_t = \sum_i |\Delta w_{i,t}| \frac{c_{i,t}^{bps}}{10000}
$$

In [ ]:
def compute_strategy_returns(daily_returns, held_weights, delta_weights, cost_bps_matrix=None):
    gross = (held_weights * daily_returns).sum(axis=1)

    if cost_bps_matrix is None:
        cost = pd.Series(0.0, index=daily_returns.index)
    else:
        cost = (delta_weights.abs() * cost_bps_matrix / 10000.0).sum(axis=1)

    net = gross - cost
    nav = (1 + net).cumprod()

    return pd.DataFrame({
        "gross_return": gross,
        "cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": delta_weights.abs().sum(axis=1),
        "gross_exposure": held_weights.abs().sum(axis=1),
        "net_exposure": held_weights.sum(axis=1),
    })

cost_bps_spread_fee = spread_component + fee_component

bt_no_cost = compute_strategy_returns(daily_returns, held_weights, delta_weights, None)
bt_spread_fee = compute_strategy_returns(daily_returns, held_weights, delta_weights, cost_bps_spread_fee)
bt_linear = compute_strategy_returns(daily_returns, held_weights, delta_weights, total_cost_linear_bps)
bt_sqrt = compute_strategy_returns(daily_returns, held_weights, delta_weights, total_cost_sqrt_bps)

backtests = {
    "no_cost": bt_no_cost,
    "spread_fee_only": bt_spread_fee,
    "linear_impact": bt_linear,
    "sqrt_impact": bt_sqrt,
}

bt_sqrt.head()

## 12. Performance under cost models

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha):
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def performance_summary(bt, name, config):
    r = bt["net_return"].dropna()
    nav = (1 + r).cumprod()
    dd, mdd = max_drawdown(nav)

    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)

    return {
        "model": name,
        "annualised_return": float(ann_return),
        "annualised_vol": float(ann_vol),
        "sharpe_proxy": float(sharpe),
        "max_drawdown": float(mdd),
        "VaR95": var,
        "CVaR95": cvar,
        "total_return": float(nav.iloc[-1] - 1.0),
        "mean_turnover": float(bt["turnover"].mean()),
        "annualised_cost_drag": float(bt["cost"].mean() * config.annualisation),
        "mean_gross_exposure": float(bt["gross_exposure"].mean()),
    }

performance_table = pd.DataFrame([
    performance_summary(bt, name, config)
    for name, bt in backtests.items()
]).sort_values("sharpe_proxy", ascending=False)

performance_table

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in backtests.items():
    plt.plot(bt.index, bt["nav"], label=name)
plt.title("NAV Under Different Cost Models")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plot_df = performance_table.sort_values("annualised_cost_drag")
plt.barh(plot_df["model"], plot_df["annualised_cost_drag"])
plt.title("Annualised Cost Drag")
plt.xlabel("Cost drag")
plt.ylabel("Cost model")
plt.show()

## 13. Cost attribution by asset

Cost contribution by asset:

$$
Cost_{i,t} = |\Delta w_{i,t}| \frac{c_{i,t}^{bps}}{10000}
$$

This identifies which assets consume the alpha.

In [ ]:
def cost_attribution(delta_weights, cost_bps_matrix, metadata, model_name):
    cost_by_asset = delta_weights.abs() * cost_bps_matrix / 10000.0

    rows = []
    for asset in cost_by_asset.columns:
        rows.append({
            "model": model_name,
            "asset": asset,
            "mean_daily_cost": cost_by_asset[asset].mean(),
            "annualised_cost": cost_by_asset[asset].mean() * config.annualisation,
            "p95_daily_cost": cost_by_asset[asset].quantile(0.95),
            "total_cost": cost_by_asset[asset].sum(),
        })

    out = pd.DataFrame(rows).merge(metadata[["asset", "asset_class", "adv_usd"]], on="asset", how="left")
    out["cost_share"] = out["total_cost"] / out["total_cost"].sum()
    return out.sort_values("total_cost", ascending=False)

cost_attr_sqrt = cost_attribution(delta_weights, total_cost_sqrt_bps, metadata, "sqrt_impact")
cost_attr_linear = cost_attribution(delta_weights, total_cost_linear_bps, metadata, "linear_impact")

cost_attr_sqrt

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = cost_attr_sqrt.sort_values("annualised_cost")
plt.barh(plot_df["asset"], plot_df["annualised_cost"])
plt.title("Annualised Cost Contribution by Asset, Square-Root Impact")
plt.xlabel("Annualised cost")
plt.ylabel("Asset")
plt.show()

## 14. Implementation shortfall simulation

We simulate fill prices from arrival prices using:

- direction;
- half-spread;
- slippage;
- impact.

For each trade:

$$
\begin{aligned}
P_{fill} &= P_{arrival} \Big[ 1 \\
&\quad + direction \cdot cost \Big]
\end{aligned}
$$

where direction is +1 for buys and -1 for sells.

Implementation shortfall is positive when execution is worse than arrival.

In [ ]:
def build_trade_list(prices, delta_weights, cost_bps_matrix, capital):
    rows = []

    for date in prices.index:
        for asset in prices.columns:
            weight_change = delta_weights.loc[date, asset]
            if abs(weight_change) < 1e-10:
                continue

            arrival_price = prices.loc[date, asset]
            direction = 1 if weight_change > 0 else -1
            trade_notional_i = abs(weight_change) * capital
            cost_bps = cost_bps_matrix.loc[date, asset]

            fill_price = arrival_price * (1 + direction * cost_bps / 10000.0)
            implementation_shortfall_bps = direction * (fill_price - arrival_price) / arrival_price * 10000.0

            rows.append({
                "date": date,
                "asset": asset,
                "direction": direction,
                "side": "BUY" if direction > 0 else "SELL",
                "weight_change": weight_change,
                "arrival_price": arrival_price,
                "fill_price": fill_price,
                "trade_notional": trade_notional_i,
                "cost_bps_model": cost_bps,
                "implementation_shortfall_bps": implementation_shortfall_bps,
                "participation": participation.loc[date, asset],
                "spread_bps": daily_spread_bps.loc[date, asset],
                "daily_volume_usd": daily_volume_usd.loc[date, asset],
            })

    return pd.DataFrame(rows)

trade_list = build_trade_list(prices, delta_weights, total_cost_sqrt_bps, config.initial_capital)

trade_list.head()

In [ ]:
tca_summary = (
    trade_list
    .groupby("asset")
    .agg(
        n_trades=("asset", "count"),
        total_notional=("trade_notional", "sum"),
        mean_is_bps=("implementation_shortfall_bps", "mean"),
        notional_weighted_is_bps=("implementation_shortfall_bps", lambda x: np.average(x, weights=trade_list.loc[x.index, "trade_notional"])),
        p95_is_bps=("implementation_shortfall_bps", lambda x: x.quantile(0.95)),
        mean_participation=("participation", "mean"),
        p95_participation=("participation", lambda x: x.quantile(0.95)),
    )
    .reset_index()
    .merge(metadata[["asset", "asset_class"]], on="asset", how="left")
    .sort_values("notional_weighted_is_bps", ascending=False)
)

tca_summary

## 15. Latency experiment

Latency can hurt through two channels:

1. prices move between decision and execution;
2. alpha decays while waiting.

We test execution delay from 0 to 80 minutes using intraday prices.

For each rebalance decision, compare arrival price to delayed price.

In [ ]:
def latency_shortfall_experiment(prices_daily, intraday_mid, delta_weights, config):
    # Map daily dates to first intraday timestamp of that day.
    rows = []

    for date in prices_daily.index:
        day_mask = intraday_mid.index.date == date.date()
        day_times = intraday_mid.index[day_mask]

        if len(day_times) == 0:
            continue

        arrival_time = day_times[0]
        day_intraday = intraday_mid.loc[day_times]

        for latency in config.latency_grid_minutes:
            step = min(int(np.round(latency / 5)), len(day_times) - 1)
            exec_time = day_times[step]

            for asset in prices_daily.columns:
                dq = delta_weights.loc[date, asset]
                if abs(dq) < 1e-10:
                    continue

                direction = 1 if dq > 0 else -1
                arrival_price = day_intraday.loc[arrival_time, asset]
                exec_price = day_intraday.loc[exec_time, asset]

                delay_shortfall_bps = direction * (exec_price - arrival_price) / arrival_price * 10000.0
                decay = alpha_decay_multiplier(latency, config.signal_half_life_minutes)

                rows.append({
                    "date": date,
                    "asset": asset,
                    "latency_minutes": latency,
                    "direction": direction,
                    "arrival_time": arrival_time,
                    "exec_time": exec_time,
                    "arrival_price": arrival_price,
                    "exec_price": exec_price,
                    "delay_shortfall_bps": delay_shortfall_bps,
                    "alpha_decay_multiplier": decay,
                    "trade_notional": abs(dq) * config.initial_capital,
                })

    return pd.DataFrame(rows)

latency_trades = latency_shortfall_experiment(prices, intraday_mid, delta_weights, config)

latency_summary = (
    latency_trades
    .groupby("latency_minutes")
    .agg(
        mean_delay_shortfall_bps=("delay_shortfall_bps", "mean"),
        notional_weighted_delay_shortfall_bps=("delay_shortfall_bps", lambda x: np.average(x, weights=latency_trades.loc[x.index, "trade_notional"])),
        p95_delay_shortfall_bps=("delay_shortfall_bps", lambda x: x.quantile(0.95)),
        alpha_decay_multiplier=("alpha_decay_multiplier", "mean"),
        n_trades=("asset", "count"),
    )
    .reset_index()
)

latency_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(latency_summary["latency_minutes"], latency_summary["notional_weighted_delay_shortfall_bps"], marker="o", label="delay shortfall")
plt.xlabel("Latency minutes")
plt.ylabel("Notional-weighted delay shortfall bps")
plt.title("Latency Delay Shortfall")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(latency_summary["latency_minutes"], latency_summary["alpha_decay_multiplier"], marker="o")
plt.xlabel("Latency minutes")
plt.ylabel("Alpha remaining")
plt.title("Alpha Decay with Latency")
plt.show()

## 16. Latency-adjusted expected alpha

Suppose the raw strategy has expected daily alpha:

$$
\alpha_0
$$

After latency:

$$
\begin{aligned}
\alpha_{\Delta t} &= \alpha_0 e^{-\ln(2)\Delta t/h} \\
&\quad - delayCost_{\Delta t}
\end{aligned}
$$

We estimate net latency-adjusted alpha from realised gross returns and delay shortfall.

In [ ]:
gross_daily_alpha_estimate = bt_no_cost["gross_return"].mean()

latency_alpha_rows = []
for _, row in latency_summary.iterrows():
    latency = row["latency_minutes"]
    decay = row["alpha_decay_multiplier"]
    delay_cost_return = row["notional_weighted_delay_shortfall_bps"] / 10000.0 * delta_weights.abs().sum(axis=1).mean()
    latency_alpha_rows.append({
        "latency_minutes": latency,
        "gross_daily_alpha_estimate": gross_daily_alpha_estimate,
        "alpha_decay_multiplier": decay,
        "delay_cost_return_estimate": delay_cost_return,
        "latency_adjusted_daily_alpha": gross_daily_alpha_estimate * decay - delay_cost_return,
        "latency_adjusted_ann_alpha": (gross_daily_alpha_estimate * decay - delay_cost_return) * config.annualisation,
    })

latency_alpha_report = pd.DataFrame(latency_alpha_rows)
latency_alpha_report

## 17. Participation and capacity curves

We scale capital and estimate cost drag.

If capital increases, participation increases:

$$
participation \propto capital
$$

Market impact rises nonlinearly.

Capacity is the point where net alpha becomes unattractive or risk limits are breached.

In [ ]:
def cost_and_performance_for_capital(capital, impact_model="sqrt"):
    scaled_trade_notional = delta_weights.abs() * capital
    scaled_participation = participation_rate(scaled_trade_notional, daily_volume_usd).fillna(0.0)

    if impact_model == "sqrt":
        impact = square_root_impact_bps(scaled_trade_notional, daily_volume_usd, realised_vol_ann, config)
    elif impact_model == "linear":
        impact = linear_impact_bps(scaled_participation, config)
    else:
        raise ValueError("impact_model must be sqrt or linear")

    cost_bps = (
        spread_component
        + fee_component
        + fixed_slippage_component
        + vol_slippage_component
        + impact
    )

    bt = compute_strategy_returns(daily_returns, held_weights, delta_weights, cost_bps)
    summary = performance_summary(bt, f"{impact_model}_{capital:.0f}", config)
    summary["capital"] = capital
    summary["impact_model"] = impact_model
    summary["max_participation"] = scaled_participation.max().max()
    summary["p95_participation"] = scaled_participation.stack().quantile(0.95)
    return summary

capital_grid = np.array([1e6, 2.5e6, 5e6, 10e6, 25e6, 50e6, 100e6, 250e6])
capacity_rows = []

for capital in capital_grid:
    capacity_rows.append(cost_and_performance_for_capital(capital, "sqrt"))
    capacity_rows.append(cost_and_performance_for_capital(capital, "linear"))

capacity_curve = pd.DataFrame(capacity_rows)

capacity_curve.head()

In [ ]:
plt.figure(figsize=(10, 6))
for model, sub in capacity_curve.groupby("impact_model"):
    plt.plot(sub["capital"], sub["annualised_return"], marker="o", label=model)
plt.xscale("log")
plt.axhline(0, linestyle="--")
plt.title("Capacity Curve: Annualised Return vs Capital")
plt.xlabel("Capital")
plt.ylabel("Annualised return")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
for model, sub in capacity_curve.groupby("impact_model"):
    plt.plot(sub["capital"], sub["annualised_cost_drag"], marker="o", label=model)
plt.xscale("log")
plt.title("Capacity Curve: Cost Drag vs Capital")
plt.xlabel("Capital")
plt.ylabel("Annualised cost drag")
plt.legend()
plt.show()

## 18. Participation clipping and opportunity cost

If a trade exceeds a maximum participation rate, only part of the desired trade can be executed.

Unfilled trade creates opportunity cost because the desired position is not achieved.

We approximate:

$$
filledNotional = \min(desiredNotional, maxParticipation \times ADV)
$$

$$
opportunityCost = |desiredNotional-filledNotional| \times |nextReturn|
$$

In [ ]:
def participation_clipped_costs(delta_weights, daily_volume_usd, daily_returns, capital, max_participation):
    desired_notional = delta_weights.abs() * capital
    max_fill_notional = daily_volume_usd * max_participation
    filled_notional = np.minimum(desired_notional, max_fill_notional)

    fill_ratio = (filled_notional / desired_notional.replace(0.0, np.nan)).fillna(1.0).clip(0.0, 1.0)
    unfilled_notional = desired_notional - filled_notional

    opportunity_cost = (unfilled_notional / capital * daily_returns.abs()).sum(axis=1)
    unfilled_rate = unfilled_notional.sum(axis=1) / desired_notional.sum(axis=1).replace(0.0, np.nan)

    return {
        "fill_ratio": fill_ratio,
        "unfilled_notional": unfilled_notional,
        "opportunity_cost": opportunity_cost.fillna(0.0),
        "unfilled_rate": unfilled_rate.fillna(0.0),
    }

participation_clip = participation_clipped_costs(
    delta_weights,
    daily_volume_usd,
    daily_returns,
    config.initial_capital,
    config.max_participation,
)

opportunity_cost_report = pd.DataFrame({
    "mean_unfilled_rate": [participation_clip["unfilled_rate"].mean()],
    "p95_unfilled_rate": [participation_clip["unfilled_rate"].quantile(0.95)],
    "max_unfilled_rate": [participation_clip["unfilled_rate"].max()],
    "annualised_opportunity_cost": [participation_clip["opportunity_cost"].mean() * config.annualisation],
})

opportunity_cost_report

## 19. Cost sensitivity grid

We vary:

- fixed slippage;
- square-root impact coefficient.

This shows whether strategy profitability depends on heroic execution assumptions.

In [ ]:
slippage_grid = [0.0, 0.5, 1.0, 2.0, 4.0]
sqrt_impact_grid = [0.25, 0.50, 0.65, 1.00, 1.50]

sensitivity_rows = []

for slip in slippage_grid:
    for impact_coeff in sqrt_impact_grid:
        cfg = CostLatencyConfig(
            fixed_slippage_bps=slip,
            sqrt_impact_coefficient=impact_coeff
        )

        cost_bps = (
            spread_component
            + fee_component
            + pd.DataFrame(slip, index=daily_returns.index, columns=assets)
            + vol_slippage_component
            + square_root_impact_bps(trade_notional, daily_volume_usd, realised_vol_ann, cfg)
        )

        bt = compute_strategy_returns(daily_returns, held_weights, delta_weights, cost_bps)
        perf = performance_summary(bt, f"slip_{slip}_impact_{impact_coeff}", config)
        perf["fixed_slippage_bps"] = slip
        perf["sqrt_impact_coefficient"] = impact_coeff
        sensitivity_rows.append(perf)

cost_sensitivity = pd.DataFrame(sensitivity_rows)

cost_sensitivity.head()

In [ ]:
pivot = cost_sensitivity.pivot(index="fixed_slippage_bps", columns="sqrt_impact_coefficient", values="sharpe_proxy")

plt.figure(figsize=(8, 6))
plt.imshow(pivot.values, aspect="auto")
plt.colorbar(label="Sharpe proxy")
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel("Square-root impact coefficient")
plt.ylabel("Fixed slippage bps")
plt.title("Cost Sensitivity: Sharpe Proxy")
plt.show()

## 20. TCA report

A transaction-cost-analysis report should include:

- number of trades;
- notional traded;
- implementation shortfall;
- spread cost;
- impact cost;
- opportunity cost;
- participation;
- latency shortfall;
- cost by asset and asset class.

In [ ]:
tca_overall = pd.DataFrame([{
    "n_trades": len(trade_list),
    "total_trade_notional": trade_list["trade_notional"].sum(),
    "mean_implementation_shortfall_bps": trade_list["implementation_shortfall_bps"].mean(),
    "notional_weighted_implementation_shortfall_bps": np.average(trade_list["implementation_shortfall_bps"], weights=trade_list["trade_notional"]) if len(trade_list) else np.nan,
    "p95_implementation_shortfall_bps": trade_list["implementation_shortfall_bps"].quantile(0.95),
    "mean_participation": trade_list["participation"].mean(),
    "p95_participation": trade_list["participation"].quantile(0.95),
    "annualised_cost_drag_sqrt_model": bt_sqrt["cost"].mean() * config.annualisation,
    "annualised_opportunity_cost": opportunity_cost_report["annualised_opportunity_cost"].iloc[0],
}])

tca_by_class = (
    cost_attr_sqrt
    .groupby("asset_class")
    .agg(
        annualised_cost=("annualised_cost", "sum"),
        total_cost=("total_cost", "sum"),
        cost_share=("cost_share", "sum"),
    )
    .reset_index()
    .sort_values("annualised_cost", ascending=False)
)

tca_overall, tca_by_class

## 21. Governance flags

A cost model should trigger review if:

1. net performance vanishes after costs;
2. annualised cost drag is too high;
3. p95 participation exceeds limit;
4. latency destroys alpha;
5. capacity curve turns negative near target capital;
6. implementation shortfall is extreme;
7. opportunity cost is material.

In [ ]:
base_no_cost_return = performance_table[performance_table["model"] == "no_cost"]["annualised_return"].iloc[0]
sqrt_return = performance_table[performance_table["model"] == "sqrt_impact"]["annualised_return"].iloc[0]
sqrt_cost_drag = performance_table[performance_table["model"] == "sqrt_impact"]["annualised_cost_drag"].iloc[0]
p95_participation = participation_summary["p95_participation"].max()
latency_alpha_20 = latency_alpha_report[latency_alpha_report["latency_minutes"] == 20]["latency_adjusted_ann_alpha"].iloc[0]
capacity_at_50m = capacity_curve[(capacity_curve["impact_model"] == "sqrt") & (capacity_curve["capital"] == 50e6)]["annualised_return"].iloc[0]
weighted_is = tca_overall["notional_weighted_implementation_shortfall_bps"].iloc[0]
opp_cost = opportunity_cost_report["annualised_opportunity_cost"].iloc[0]

governance_flags = pd.DataFrame([{
    "base_no_cost_ann_return": base_no_cost_return,
    "sqrt_cost_ann_return": sqrt_return,
    "sqrt_annualised_cost_drag": sqrt_cost_drag,
    "max_p95_participation": p95_participation,
    "latency_20min_ann_alpha": latency_alpha_20,
    "capacity_50m_ann_return": capacity_at_50m,
    "notional_weighted_is_bps": weighted_is,
    "annualised_opportunity_cost": opp_cost,
    "flag_costs_erase_more_than_50pct_return": bool((base_no_cost_return - sqrt_return) > 0.50 * abs(base_no_cost_return)),
    "flag_cost_drag_above_5pct": bool(sqrt_cost_drag > 0.05),
    "flag_p95_participation_above_max": bool(p95_participation > config.max_participation),
    "flag_latency_20min_alpha_negative": bool(latency_alpha_20 < 0),
    "flag_capacity_50m_negative": bool(capacity_at_50m < 0),
    "flag_weighted_is_above_25bps": bool(weighted_is > 25),
    "flag_opportunity_cost_above_2pct": bool(opp_cost > 0.02),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_costs_erase_more_than_50pct_return",
        "flag_cost_drag_above_5pct",
        "flag_p95_participation_above_max",
        "flag_latency_20min_alpha_negative",
        "flag_capacity_50m_negative",
        "flag_weighted_is_above_25bps",
        "flag_opportunity_cost_above_2pct",
    ]
].any(axis=1)

governance_flags

## 22. Audit checks

Numerical checks:

1. cost components are non-negative;
2. participation is finite;
3. cost drag increases from no-cost to cost models;
4. capacity cost increases with capital;
5. alpha decay decreases with latency;
6. trade list has positive notional;
7. no output contains infinite values.

In [ ]:
audit_rows = []

all_components_nonnegative = all([
    (spread_component >= 0).all().all(),
    (fee_component >= 0).all().all(),
    (fixed_slippage_component >= 0).all().all(),
    (vol_slippage_component >= 0).all().all(),
    (linear_impact_component >= 0).all().all(),
    (sqrt_impact_component >= 0).all().all(),
])

audit_rows.append({
    "check": "cost_components_nonnegative",
    "value": bool(all_components_nonnegative),
    "passed": bool(all_components_nonnegative),
})

participation_finite = bool(np.isfinite(participation.fillna(0.0).to_numpy()).all())
audit_rows.append({
    "check": "participation_finite",
    "value": participation_finite,
    "passed": participation_finite,
})

cost_drag_order_ok = (
    performance_table.set_index("model").loc["sqrt_impact", "annualised_cost_drag"]
    >= performance_table.set_index("model").loc["spread_fee_only", "annualised_cost_drag"]
    >= performance_table.set_index("model").loc["no_cost", "annualised_cost_drag"]
)

audit_rows.append({
    "check": "cost_drag_order_reasonable",
    "value": bool(cost_drag_order_ok),
    "passed": bool(cost_drag_order_ok),
})

cap_sqrt = capacity_curve[capacity_curve["impact_model"] == "sqrt"].sort_values("capital")
capacity_cost_monotone = bool(cap_sqrt["annualised_cost_drag"].is_monotonic_increasing)
audit_rows.append({
    "check": "capacity_cost_increases_with_capital",
    "value": capacity_cost_monotone,
    "passed": capacity_cost_monotone,
})

alpha_decay_decreasing = bool(latency_summary["alpha_decay_multiplier"].is_monotonic_decreasing)
audit_rows.append({
    "check": "alpha_decay_decreases_with_latency",
    "value": alpha_decay_decreasing,
    "passed": alpha_decay_decreasing,
})

trade_notional_positive = bool((trade_list["trade_notional"] > 0).all()) if len(trade_list) else True
audit_rows.append({
    "check": "trade_notional_positive",
    "value": trade_notional_positive,
    "passed": trade_notional_positive,
})

finite_outputs = bool(
    np.isfinite(performance_table.select_dtypes(include=[float, int]).to_numpy()).all()
    and np.isfinite(cost_component_summary.select_dtypes(include=[float, int]).to_numpy()).all()
)
audit_rows.append({
    "check": "key_outputs_finite",
    "value": finite_outputs,
    "passed": finite_outputs,
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 23. Practical checklist for transaction-cost modelling

Before trusting cost-adjusted performance:

1. **Separate cost components**  
   Spread, fees, slippage, impact, delay, and opportunity cost should be distinct.

2. **Measure participation**  
   High returns at impossible participation are not useful.

3. **Model impact nonlinearly**  
   Large orders rarely scale linearly without penalty.

4. **Stress spreads and volume**  
   Liquidity disappears when it matters.

5. **Check latency**  
   Fast-decay alpha cannot tolerate slow execution.

6. **Compare pre-cost and post-cost results**  
   If all alpha disappears, the strategy is not tradable.

7. **Run capacity curves**  
   Estimate when capital breaks the strategy.

8. **Report implementation shortfall**  
   Fill quality should be visible by asset and class.

9. **Include opportunity cost**  
   Unfilled orders matter.

10. **Govern assumptions**  
   Cost parameters should be conservative, documented, and stress-tested.

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/transaction_costs_slippage_latency_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "daily_returns": output_dir / "daily_returns.csv",
    "prices": output_dir / "prices.csv",
    "daily_spread_bps": output_dir / "daily_spread_bps.csv",
    "daily_volume_usd": output_dir / "daily_volume_usd.csv",
    "intraday_mid": output_dir / "intraday_mid.csv",
    "intraday_spread": output_dir / "intraday_spread.csv",
    "intraday_volume": output_dir / "intraday_volume.csv",
    "metadata": output_dir / "metadata.csv",
    "regime_info": output_dir / "regime_info.csv",
    "signal": output_dir / "signal.csv",
    "target_weights": output_dir / "target_weights.csv",
    "held_weights": output_dir / "held_weights.csv",
    "delta_weights": output_dir / "delta_weights.csv",
    "participation": output_dir / "participation.csv",
    "participation_summary": output_dir / "participation_summary.csv",
    "cost_component_summary": output_dir / "cost_component_summary.csv",
    "total_cost_linear_bps": output_dir / "total_cost_linear_bps.csv",
    "total_cost_sqrt_bps": output_dir / "total_cost_sqrt_bps.csv",
    "bt_no_cost": output_dir / "backtest_no_cost.csv",
    "bt_spread_fee": output_dir / "backtest_spread_fee.csv",
    "bt_linear": output_dir / "backtest_linear_impact.csv",
    "bt_sqrt": output_dir / "backtest_sqrt_impact.csv",
    "performance_table": output_dir / "performance_table.csv",
    "cost_attr_sqrt": output_dir / "cost_attribution_sqrt.csv",
    "trade_list": output_dir / "trade_list.csv",
    "tca_summary": output_dir / "tca_summary.csv",
    "latency_summary": output_dir / "latency_summary.csv",
    "latency_alpha_report": output_dir / "latency_alpha_report.csv",
    "capacity_curve": output_dir / "capacity_curve.csv",
    "opportunity_cost_report": output_dir / "opportunity_cost_report.csv",
    "cost_sensitivity": output_dir / "cost_sensitivity.csv",
    "tca_overall": output_dir / "tca_overall.csv",
    "tca_by_class": output_dir / "tca_by_class.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

daily_returns.to_csv(paths["daily_returns"])
prices.to_csv(paths["prices"])
daily_spread_bps.to_csv(paths["daily_spread_bps"])
daily_volume_usd.to_csv(paths["daily_volume_usd"])
intraday_mid.to_csv(paths["intraday_mid"])
intraday_spread.to_csv(paths["intraday_spread"])
intraday_volume.to_csv(paths["intraday_volume"])
metadata.to_csv(paths["metadata"], index=False)
regime_info.to_csv(paths["regime_info"])
signal.to_csv(paths["signal"])
target_weights.to_csv(paths["target_weights"])
held_weights.to_csv(paths["held_weights"])
delta_weights.to_csv(paths["delta_weights"])
participation.to_csv(paths["participation"])
participation_summary.to_csv(paths["participation_summary"], index=False)
cost_component_summary.to_csv(paths["cost_component_summary"], index=False)
total_cost_linear_bps.to_csv(paths["total_cost_linear_bps"])
total_cost_sqrt_bps.to_csv(paths["total_cost_sqrt_bps"])
bt_no_cost.to_csv(paths["bt_no_cost"])
bt_spread_fee.to_csv(paths["bt_spread_fee"])
bt_linear.to_csv(paths["bt_linear"])
bt_sqrt.to_csv(paths["bt_sqrt"])
performance_table.to_csv(paths["performance_table"], index=False)
cost_attr_sqrt.to_csv(paths["cost_attr_sqrt"], index=False)
trade_list.to_csv(paths["trade_list"], index=False)
tca_summary.to_csv(paths["tca_summary"], index=False)
latency_summary.to_csv(paths["latency_summary"], index=False)
latency_alpha_report.to_csv(paths["latency_alpha_report"], index=False)
capacity_curve.to_csv(paths["capacity_curve"], index=False)
opportunity_cost_report.to_csv(paths["opportunity_cost_report"], index=False)
cost_sensitivity.to_csv(paths["cost_sensitivity"], index=False)
tca_overall.to_csv(paths["tca_overall"], index=False)
tca_by_class.to_csv(paths["tca_by_class"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "transaction_costs_slippage_latency_outputs",
    "schema_version": "transaction_costs_slippage_latency_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "assets": assets,
    "performance_table": performance_table.to_dict(orient="records"),
    "tca_overall": tca_overall.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "Cost components are separated into spread, fees, fixed slippage, volatility slippage, and impact.",
        "Both linear and square-root impact models are compared.",
        "Implementation shortfall is measured at trade level.",
        "Latency experiments estimate delay shortfall and alpha decay.",
        "Capacity curves scale capital and recompute cost-adjusted performance.",
        "Participation clipping approximates partial fills and opportunity cost.",
        "This notebook is a cost-model research layer for later event-driven and production-style backtests."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["performance_table"], paths["tca_overall"], paths["capacity_curve"], paths["governance_flags"], paths["manifest"]

## 25. Limitations

### Synthetic execution data

The notebook uses synthetic spreads, volumes, and intraday prices. Real execution data can be messier and more path-dependent.

### Simplified impact

Linear and square-root impact are approximations. Actual impact depends on order type, volatility, liquidity, urgency, venue, queue position, and market state.

### No full order book

The notebook does not model queue position or depth beyond aggregate volume assumptions.

### Simple alpha decay

Alpha decay is modelled exponentially. Real decay can be nonlinear and regime-dependent.

### Simplified opportunity cost

Opportunity cost is approximated using next-period absolute returns. A production model should use actual unfilled order behaviour.

### No broker fee schedule

Commission and fees are simple bps assumptions.

### No live TCA data

Real TCA should compare actual fills to arrival, VWAP, TWAP, close, and decision benchmarks.

## 26. Research frontier and extensions

Important extensions include:

1. venue-specific execution models;
2. queue-position simulation;
3. order-book depth and hidden liquidity;
4. nonlinear transient and permanent impact;
5. Almgren-Chriss execution scheduling;
6. adaptive participation algorithms;
7. live fill TCA;
8. adverse-selection modelling;
9. execution alpha versus investment alpha;
10. Chinese futures L1/L2 slippage with night sessions, tick sizes, price limits, and exchange fee schedules.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_04_walk_forward_hedge_validation**  
   Validate hedges after realistic costs.

2. **05_05_purged_cross_validation_backtesting**  
   Avoid leakage in model validation.

3. **05_08_strategy_capacity_and_market_impact**  
   Build deeper market-impact and capacity models.

4. **05_09_event_driven_backtest_framework**  
   Plug these costs into a modular event system.

5. **05_11_live_shadow_mode_monitoring**  
   Compare predicted costs with live/paper fills.

6. **06_06_execution_algorithms_twap_vwap_pov**  
   Turn cost models into execution algorithms.

## 28. Summary

This notebook implemented transaction-cost, slippage, and latency research.

It showed how to:

1. simulate daily and intraday liquidity data;
2. build strategy target trades;
3. estimate participation rates;
4. separate half-spread, fees, slippage, and impact;
5. compare linear and square-root impact;
6. compute pre-cost and post-cost returns;
7. attribute cost by asset and class;
8. simulate implementation shortfall;
9. run latency and alpha-decay experiments;
10. build capacity curves;
11. estimate opportunity cost from participation limits;
12. run cost sensitivity grids;
13. produce TCA reports;
14. create governance flags and audit checks;
15. save outputs and metadata.

The key computational takeaway:

> Cost modelling is not one number; it is a decomposition of spread, fees, slippage, impact, latency, and opportunity cost.

The key financial takeaway:

> A strategy with small alpha and high turnover is not really an alpha strategy — it is a liquidity donation programme.

## 29. Further reading

- Almgren and Chriss on optimal execution.
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Cartea, Jaimungal, and Penalva, *Algorithmic and High-Frequency Trading.*
- Grinold and Kahn, *Active Portfolio Management.*
- López de Prado, *Advances in Financial Machine Learning.*
- Institutional transaction-cost-analysis literature on implementation shortfall, market impact, and capacity.